[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/04_agent_loop.ipynb)

# 04. The Agent Loop

**Part 1: The Loop** · about 20 minutes

Here is the idea this whole course is built on. In notebook 03 we did one round trip by hand. An agent is that same round trip inside a `while` loop:

```
                ┌──────────────────────┐
                │ send messages + tools│
                └──────────┬───────────┘
                           ▼
                    did the model ask
                     to call a tool?
                     │             │
                    yes            no ──► done, that's the answer
                     ▼
               run the tool(s)
               append the results
                     │
                     └──► back to the top
```

The important shift: **the model decides what to do next, not your code.** You don't write the steps. You write the tools and the stopping rules.

**By the end you will have:**

- a working agent loop
- a clear view of how its cost grows with each step
- safeguards against the two classic failures: looping forever and crashing on bad tools

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import time

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"


def call_model(messages, tools):
    payload = {"model": llm.model, "messages": messages, "tools": tools}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()

## 1. Give it something to explore

Our test task needs several steps, and the agent shouldn't know the steps in advance. It has a small set of notes held in a Python dict, so nothing here can touch your real files. The answer is spread across two of them, and the agent has to discover that on its own.

In [3]:
NOTES = {
    "standup.md": "Standup notes. The launch deadline moved to March 14 because of the audit.",
    "owners.md": "Launch owner: Priya. Backend owner: Sam.",
    "lunch.md": "The team voted for dosa on Friday.",
}


def list_files():
    return json.dumps(sorted(NOTES))


def read_file(path):
    if path not in NOTES:
        raise FileNotFoundError(f"No file named {path!r}")
    return NOTES[path]


REGISTRY = {"list_files": list_files, "read_file": read_file}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List the names of all available note files.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read one note file by its exact name.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string", "description": "The file name"}},
                "required": ["path"],
            },
        },
    },
]


def run_tool_call(call):
    """The dispatcher from notebook 03: tool failures become text for the model."""
    name = call["function"]["name"]
    if name not in REGISTRY:
        return f"Error: unknown tool {name!r}. Available tools: {sorted(REGISTRY)}"
    try:
        arguments = json.loads(call["function"]["arguments"] or "{}")
        return str(REGISTRY[name](**arguments))
    except Exception as error:
        return f"Error: {type(error).__name__}: {error}"

## 2. The loop

Read this slowly. It's short, and everything else in the course attaches to it.

We take `model` as a parameter so we can swap in a fake one for testing later, and every run is limited by `max_steps`. An agent without a limit is an unattended loop that spends money.

In [4]:
def run_agent(question, model=call_model, max_steps=8, verbose=True):
    messages = [
        {"role": "system", "content": "Answer using the tools. Read the files you need, then reply briefly."},
        {"role": "user", "content": question},
    ]
    steps = []

    for step in range(1, max_steps + 1):
        data = model(messages, TOOLS)
        message = data["choices"][0]["message"]
        messages.append(message)
        steps.append({"step": step, "messages": len(messages) - 1, "usage": data.get("usage") or {}})

        tool_calls = message.get("tool_calls") or []
        if not tool_calls:  # no tool requested, so the model is finished
            return {"status": "done", "answer": message.get("content"), "messages": messages, "steps": steps}

        for call in tool_calls:
            result = run_tool_call(call)
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            if verbose:
                print(f"step {step}: {call['function']['name']}({call['function']['arguments']})")
                print(f"        -> {result[:70]}")

    return {"status": "max_steps", "answer": None, "messages": messages, "steps": steps}

## 3. Run it

We never told it to list the files first, or which files to read.

In [5]:
run = run_agent("What is the launch deadline, and who owns the launch?")

print()
print("Status:", run["status"])
print("Answer:", run["answer"])

step 1: list_files({})
        -> ["lunch.md", "owners.md", "standup.md"]


step 2: read_file({"path": "lunch.md"})
        -> The team voted for dosa on Friday.


step 3: read_file({"path": "owners.md"})
        -> Launch owner: Priya. Backend owner: Sam.
step 3: read_file({"path": "standup.md"})
        -> Standup notes. The launch deadline moved to March 14 because of the au



Status: done
Answer: The launch deadline is March 14, and the owner is Priya.


In [6]:
# Checkpoint: the offline script has to combine facts from two different files.
if llm.offline:
    assert run["status"] == "done"
    assert "March 14" in run["answer"] and "Priya" in run["answer"]
print("Checkpoint passed.")

Checkpoint passed.


Notice that one step can contain more than one tool call. Models can ask for several things at once (*parallel tool calls*), which is why the loop goes through `tool_calls` rather than taking only the first. Every call must get a result, or the next request will be rejected.

## 4. What the model actually saw

Here's the list that we resent on each step. It's the agent's entire memory.

In [7]:
for message in run["messages"]:
    role = message["role"]
    if message.get("tool_calls"):
        names = [c["function"]["name"] for c in message["tool_calls"]]
        print(f"{role:<10} wants to call {names}")
    else:
        print(f"{role:<10} {(message.get('content') or '')[:70]}")

system     Answer using the tools. Read the files you need, then reply briefly.
user       What is the launch deadline, and who owns the launch?
assistant  wants to call ['list_files']
tool       ["lunch.md", "owners.md", "standup.md"]
assistant  wants to call ['read_file']
tool       The team voted for dosa on Friday.
assistant  wants to call ['read_file', 'read_file']
tool       Launch owner: Priya. Backend owner: Sam.
tool       Standup notes. The launch deadline moved to March 14 because of the au
assistant  The launch deadline is March 14, and the owner is Priya.


## 5. The cost of a loop

In notebook 01, input tokens grew with each turn of a chat. An agent is the same thing, except it might take 50 turns and tool results can be enormous.

In [8]:
print("step | messages sent | input tokens")
print("-----+---------------+-------------")
total_input = 0
for step in run["steps"]:
    tokens = step["usage"].get("prompt_tokens", 0)
    total_input += tokens
    print(f"{step['step']:>4} | {step['messages']:>13} | {tokens:>6}  {'#' * (tokens // 20)}")

final_size = run["steps"][-1]["usage"].get("prompt_tokens", 0)
print(f"\nTotal input tokens billed: {total_input}")
print(f"Size of the final request: {final_size}")

step | messages sent | input tokens
-----+---------------+-------------
   1 |             2 |    152  #######
   2 |             4 |    215  ##########
   3 |             6 |    281  ##############
   4 |             9 |    411  ####################

Total input tokens billed: 1059
Size of the final request: 411


The total billed is much larger than the final conversation, because early messages get paid for again on every later step. If each step adds about the same amount, total cost grows roughly with the **square** of the number of steps.

This is why serious agents care so much about keeping tool results short, and why Parts 3 and 4 cover compaction and caching.

## 6. Failure one: the agent that never stops

Models can get stuck, repeating the same action over and over. To test our guard we don't need a real model to misbehave. We can write a fake one that always asks for the same tool. Testing with a fake model like this is a technique you'll use constantly, since it's free, fast and repeatable.

In [9]:
def stubborn_model(messages, tools):
    """A fake model that never stops asking for the same tool."""
    return {
        "choices": [{
            "message": {
                "role": "assistant",
                "content": None,
                "tool_calls": [{
                    "id": f"call_{len(messages)}",
                    "type": "function",
                    "function": {"name": "list_files", "arguments": "{}"},
                }],
            }
        }]
    }


stuck = run_agent("Anything", model=stubborn_model, max_steps=3, verbose=False)

print("Status:", stuck["status"])
print("Steps: ", len(stuck["steps"]))

assert stuck["status"] == "max_steps"
assert len(stuck["steps"]) == 3
print("Checkpoint passed.")

Status: max_steps
Steps:  3
Checkpoint passed.


## 7. Failure two: the tool that fails

Next, a model that asks for a file that doesn't exist. If `read_file` crashed our program, the run would die. Because the dispatcher returns the error as text, the model gets the chance to read it and recover. This fake model does just that.

In [10]:
def recovering_model(messages, tools):
    """A fake model that makes a mistake, sees the error, and corrects itself."""
    last = messages[-1]

    def ask(path):
        return {"choices": [{"message": {
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": f"call_{len(messages)}",
                "type": "function",
                "function": {"name": "read_file", "arguments": json.dumps({"path": path})},
            }],
        }}]}

    if last["role"] == "user":
        return ask("owner.md")  # a typo: the real file is owners.md
    if last["content"].startswith("Error"):
        return ask("owners.md")  # it read the error and fixed the name
    return {"choices": [{"message": {"role": "assistant", "content": f"Found it: {last['content']}"}}]}


recovered = run_agent("Who owns the launch?", model=recovering_model)

print("\nStatus:", recovered["status"])
print("Answer:", recovered["answer"])

assert recovered["status"] == "done"
assert "Priya" in recovered["answer"]
print("Checkpoint passed.")

step 1: read_file({"path": "owner.md"})
        -> Error: FileNotFoundError: No file named 'owner.md'
step 2: read_file({"path": "owners.md"})
        -> Launch owner: Priya. Backend owner: Sam.

Status: done
Answer: Found it: Launch owner: Priya. Backend owner: Sam.
Checkpoint passed.


The agent recovered from a mistake with no special error-handling logic, because the model handled the error the same way it handles any other text. **Good error messages are part of your agent's design.** Compare `No file named 'owner.md'` with a bare `Error 2`: only one of them helps it recover.

## Recap

- An agent is a model call in a loop: run the requested tools, append the results, repeat until no tools are requested.
- The model picks the steps, and you control the tools and the limits.
- Input tokens are paid for again on every step, so long runs get expensive quickly.
- Always cap the steps, and return tool failures to the model as text.
- Fake models let you test the loop for free.

## Exercises

1. Add a `max_tokens_total` budget that stops the run once the billed input tokens pass a limit.
2. Make `read_file` return a giant string, 100 times longer. What happens to the cost table? How would you fix that?
3. Detect a stuck agent: if the model repeats the identical tool call three times in a row, stop early with `status: "stuck"`. Test it with `stubborn_model`.

**Next:** [05. Your First Real Agent](05_first_real_agent.ipynb), where this loop fixes real bugs in real code.